# Task 9: BERT Question Answering

Finetuning BERT to predict start and end positions of answer spans.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

## QA Model

BERT QA predicts:
- **Start position**: Where the answer begins
- **End position**: Where the answer ends

In [ ]:
class BERTForQuestionAnswering(nn.Module):
    """BERT for Question Answering."""
    
    def __init__(self, vocab_size, d_model=768, num_heads=12, num_layers=12, d_ff=3072):
        super().__init__()
        self.bert = BERTEncoder(vocab_size, d_model, num_heads, num_layers, d_ff)
        
        # Output: start and end positions
        self.qa_outputs = nn.Linear(d_model, 2)
        
    def forward(self, token_ids, start_positions=None, end_positions=None):
        sequence_output, _ = self.bert(token_ids)
        
        logits = self.qa_outputs(sequence_output)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)
        
        loss = None
        if start_positions is not None and end_positions is not None:
            # SQuAD loss
            loss_fct = nn.CrossEntropyLoss()
            start_loss = loss_fct(start_logits, start_positions)
            end_loss = loss_fct(end_logits, end_positions)
            loss = (start_loss + end_loss) / 2
        
        return loss, start_logits, end_logits


# Minimal encoder for demo
class BERTEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_heads=12, num_layers=12, d_ff=3072):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, d_model)
        self.encoder_layers = nn.ModuleList([TransformerEncoderBlock(d_model, num_heads, d_ff) 
                                              for _ in range(num_layers)])
        
    def forward(self, token_ids):
        x = self.embeddings(token_ids)
        for layer in self.encoder_layers:
            x = layer(x)
        return x, x[:, 0]


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, d_ff):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, d_ff), nn.GELU(), nn.Linear(d_ff, embed_dim))
        
    def forward(self, x):
        x = self.norm1(x + self.attn(x, x, x)[0])
        x = self.norm2(x + self.ffn(x))
        return x

# Test
model = BERTForQuestionAnswering(vocab_size=30000, d_model=128, num_heads=4, num_layers=2, d_ff=256)
token_ids = torch.randint(0, 30000, (2, 64))

loss, start_logits, end_logits = model(token_ids)
print(f"Loss: {loss.item():.4f}")
print(f"Start logits: {start_logits.shape}")
print(f"End logits: {end_logits.shape}")

In [ ]:
def predict_answer(model, question_ids, context_ids):
    """Extract answer span from context."""
    model.eval()
    with torch.no_grad():
        # Combine [CLS] question [SEP] context [SEP]
        input_ids = torch.cat([question_ids, context_ids], dim=1)
        
        _, start_logits, end_logits = model(input_ids)
        
        # Get best start and end
        start_idx = start_logits[0].argmax().item()
        end_idx = end_logits[0].argmax().item()
        
        # Ensure end >= start
        if end_idx < start_idx:
            end_idx = start_idx
        
    return start_idx, end_idx

# Demo
question = torch.tensor([[101, 2054, 2003, 2019, 2890, 102]])  # "Who is president"
context = torch.tensor([[101, 2003, 2023, 10176, 1996, 2447, 102] + [0]*56])  # "The president is Biden."

start, end = predict_answer(model, question, context)
print(f"Predicted answer positions: start={start}, end={end}")

## Summary
- ✅ Predict start and end positions
- ✅ Use span extraction (not classification)
- ✅ SQuAD-style training

## Next: [Task 10: Inference & Model Serving](../task-10-inference-serving/overview.html)